# Ejercicio final de la semana 1

Para demostrar que estás familiarizado con la API de OpenAI y también con Ollama, crea una herramienta que responda a una pregunta técnica
y la explique. ¡Esta es una herramienta que podrás usar durante el curso!

In [2]:
# imports

import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI


In [3]:
# constantes

OLLAMA_API = "http://localhost:11434/api/chat"
HEADERS = {"Content-Type": "application/json"}
MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'


In [4]:
# Configura el entorno

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key[:8]=='sk-proj-':
    print("La clave de API parece buena")
else:
    print("¿Puede haber un problema con tu clave API? ¡Visita el cuaderno de resolución de problemas!")

openai = OpenAI()

La clave de API parece buena


In [5]:
# Aquí está la pregunta; escribe sobre esto para preguntar algo nuevo

system_prompt = "Necesito que actues como un programador experto en Python. \
Responde lo más detallado posible, dando ejemplos. \
Responde en formato markdown.\n"

user_prompt = """
Explica qué hace este código y por qué:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [8]:
# Haz que gpt-4o-mini responda, con streaming

def python_expert():
    stream = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        stream=True
    )

    response = ""
    display_handle = display(Markdown(""), display_id=True)

    for chunk in stream:
        delta = chunk.choices[0].delta
        content = getattr(delta, 'content', '')
        if content:
            response += content
            update_display(Markdown(response), display_id=display_handle.display_id)

    return response



# Llamada a la función
final_response = python_expert()



Claro, analicemos el código línea por línea y detalle qué hace y por qué. 

### Análisis del Código

El fragmento que compartiste es el siguiente:
```python
yield from {book.get("author") for book in books if book.get("author")}
```

1. **Comprensión de Conjunto**:
   - `{book.get("author") for book in books if book.get("author")}` es una **comprensión de conjunto** en Python. Esto significa que se está creando un conjunto (un tipo de colección que no permite duplicados) que contiene todos los autores de los libros de `books`.

   - Dentro de la comprensión de conjunto:
     - `book.get("author")` intenta obtener el valor asociado con la clave `"author"` en cada objeto `book`.
     - `for book in books` itera sobre una colección llamada `books`.
     - `if book.get("author")` actúa como un filtro, asegurándose de que solo se incluyan en el conjunto aquellos valores que no sean `None` (es decir, que realmente existe un autor).

2. **yield from**:
   - La palabra clave `yield from` se utiliza dentro de una función generadora. Esto permite que la función produzca valores de otro iterable, en este caso, el conjunto de autores que hemos creado.

   - `yield from` iterará sobre cada elemento del iterable proporcionado, que es el conjunto de autores únicos, y devolverá cada autor uno por uno.

### Ejemplo Completo

Imaginemos que tenemos una lista de libros representada como una lista de diccionarios y necesitamos obtener una lista de autores únicos:

```python
def get_unique_authors(books):
    yield from {book.get("author") for book in books if book.get("author")}

# Lista de libros
books = [
    {"title": "Libro 1", "author": "Autor A"},
    {"title": "Libro 2", "author": "Autor B"},
    {"title": "Libro 3", "author": None},
    {"title": "Libro 4", "author": "Autor A"},
    {"title": "Libro 5", "author": "Autor C"},
]

# Consumiendo la función
authors_generator = get_unique_authors(books)

for author in authors_generator:
    print(author)
```

### Salida Esperada

El resultado del código anterior sería:
```
Autor A
Autor C
Autor B
```

### Resumen

1. **Funcionalidad**: El código extrae los autores únicos de una lista de libros, ignorando aquellos libros que no tienen un autor definido (es decir, cuyo valor de "author" es `None`).
2. **Eficiencia**: Usar un conjunto (`set`) asegura que no haya duplicados en los autores devueltos.
3. **Uso de `yield from`**: Permite que la función sea un generador que produce cada autor de manera eficiente, en lugar de devolver una lista completa en una sola vez, lo cual puede economizar memoria.

Esta construcción de Python es bastante útil en situaciones donde necesitas procesar colecciones y generar elementos de forma eficiente.

In [12]:
# Haz que Llama 3.2 responda

messages = [
    {
        "role": "user",
        "content": (
            "Necesito que actúes como un programador experto en Python. "
            "Responde lo más detallado posible, dando ejemplos. "
            "Responde en formato markdown. "
            "Explica qué hace este código y por qué: "
            "yield from {book.get('author') for book in books if book.get('author')}"
        )
    }
]

payload = {
    "model": MODEL_LLAMA,
    "messages": messages,
    "stream": False
}

response = requests.post(OLLAMA_API, json=payload, headers=HEADERS)

# Procesar la respuesta JSON
response_json = response.json()

# Extraer el contenido de la respuesta
message_content = response_json.get('message', {}).get('content', 'No content received')

# Mostrar en formato Markdown
display(Markdown(message_content))

**Análisis del Código**
======================

El código que se proporciona es una expresión `yield` dentro de un contexto de iteración. A continuación, se explica cómo funciona y por qué.

**Explicación de la Expresión**
---------------------------

La expresión se puede descomponer en las siguientes partes:

*   `yield`: Es el keyword que indica que la función produce una serie de valores en lugar de un solo valor.
*   `{book.get('author') for book in books if book.get('author')}`: Es una expresión generada (generator expression) que produce una secuencia de valores.

    *   `for book in books`: Esta parte del código itera sobre la variable `books` y asigna cada elemento en la lista a la variable `book`.
    *   `if book.get('author')`: Esto es una condición que solo se cumple si el valor `author` existe dentro de cada diccionario `book`. De lo contrario, no se procesarán los valores.
    *   `book.get('author')`: Este es el valor que se está generando para cada iteración.

**Funcionamiento**
----------------

Cuando se ejecuta la expresión `yield from {book.get('author') for book in books if book.get('author')}`, aquí hay lo siguiente:

1.  Se pasa por iteración sobre `books`. Cada diccionario en esta lista tiene un valor de `author`.
2.  Si el `author` existe en ese diccionario, se procesa.
3.  El valor del autor se produce y se envía al generador que lo recoge.
4.  Se repite este proceso para cada elemento de la lista `books`.

**Por qué es útil**
------------------

El código es útil cuando necesitas iterar sobre los valores de un diccionario en una lista, pero deseas procesarlos individualmente y generar una secuencia de estos valores.

En este caso, el código se utiliza para recuperar solo los autores de los libros que tienen un autor asociado. Por ejemplo:

```python
books = [
    {"title": "Harry Potter", "author": "J.K. Rowling"},
    {"title": "El Señor de los Anillos", "author": None},
    {"title": "El mago de Oz", "author": "L. Frank Baum"}
]

autores_procesados = yield from {book.get('author') for book in books if book.get('author')}
print(autores_procesados)  # Salida: ['J.K. Rowling', None, 'L. Frank Baum']
```

**Conclusión**
--------------

En resumen, la expresión `yield from {book.get('author') for book in books if book.get('author')}` es una forma de procesar y generar secuencias de valores de manera eficiente cuando se trabaja con diccionarios. Esta expresión es útil en situaciones donde debes iterar sobre datos y realizar operaciones de procesamiento individualmente.